GRUPO:



*   Thiago Corrêa Brandão
*   Isabella Vieira
*   João Pedro Menezes
*   Breno Alcaraz
*   Fabiano Amorim









# Carregando os dados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

# Carregar o dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "stance_climate")

# Extrair informações sobre os tamanhos dos conjuntos
train_size = len(dataset['train'])
test_size = len(dataset['test'])
validation_size = len(dataset['validation'])
total_size = train_size + test_size + validation_size

# Calcular percentuais
train_percent = (train_size / total_size) * 100
test_percent = (test_size / total_size) * 100
validation_percent = (validation_size / total_size) * 100

# Dados para o gráfico
categories = ['Treino', 'Teste', 'Validação']
sizes = [train_size, test_size, validation_size]
percents = [train_percent, test_percent, validation_percent]
colors = ['#2E86AB', '#A23B72', '#F18F01']

# Criar o gráfico de barras
plt.figure(figsize=(12, 8))

# Gráfico de barras
bars = plt.bar(categories, sizes, color=colors, edgecolor='black', alpha=0.8)

# Adicionar valores nas barras
for i, (size, percent) in enumerate(zip(sizes, percents)):
    plt.text(i, size + 5, f'{size}\n({percent:.1f}%)',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Personalizar o gráfico
plt.ylabel('Número de Amostras', fontsize=12, fontweight='bold')
plt.xlabel('Conjunto de Dados', fontsize=12, fontweight='bold')
plt.title('Distribuição do Dataset Stance Climate\npor Conjunto de Dados',
          fontsize=16, fontweight='bold', pad=20)

# Ajustar limites do eixo Y para melhor visualização
plt.ylim(0, max(sizes) * 1.15)

# Adicionar grade para melhor leitura
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Remover bordas desnecessárias
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Ajustar layout
plt.tight_layout()

# Mostrar o gráfico
plt.show()

In [ ]:
# observar alguns registros aleatorios no treino
import random
random_index = random.randint(0, len(dataset['train']) - 1)
random_example = dataset['train'][random_index]
print(f"Índice: {random_index}")
print(f"Tweet: {random_example['text']}")
print(f"Label: {random_example['label']}")

# Pré-processamento

In [ ]:
# Aqui iremos aplicar algumas técnicas de
# Mapeamento de labels para stance_climate (apenas none e favor)
label_mapping = {
    0: "none",   # Nenhum stance
    1: "favor"   # A favor de ações climáticas
}

# Filtrar apenas exemplos com label 0 (none) ou 2 (favor),
# remapeando 2 -> 1 para manter IDs contíguos
def filter_and_remap(example):
    return example['label'] in [0, 2]

def remap_label(example):
    example['label'] = 0 if example['label'] == 0 else 1
    return example

dataset = dataset.filter(filter_and_remap)
dataset = dataset.map(remap_label)

# Função para adicionar labels textuais
def add_stance_labels(example):
    example['stance_text'] = label_mapping[example['label']]
    return example

# Aplicar o mapeamento
dataset = dataset.map(add_stance_labels)

# Analisar distribuição por split
for split in ['train', 'test', 'validation']:
    print(f'\n--- Distribuição no {split} ---')
    split_data = dataset[split]
    labels = [ex['label'] for ex in split_data]
    for label_num in sorted(set(labels)):
        count = labels.count(label_num)
        percentage = (count / len(split_data)) * 100
        print(f'  {label_mapping[label_num]}: {count} exemplos ({percentage:.1f}%)')
    print(f'  Total: {len(split_data)}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

splits = ['treino', 'teste', 'validação']
labels = ['none', 'favor']

# Dados após remoção da classe 'against'
data = {
    'treino': [151, 191],
    'teste':  [35,  123],
    'validação': [17, 23]
}

percentuais = {
    'treino':    [44.2, 55.8],
    'teste':     [22.1, 77.9],
    'validação': [42.5, 57.5]
}

colors = ['#FF6B6B', '#45B7D1']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, split in enumerate(splits):
    valores = data[split]
    percents = percentuais[split]
    bars = axes[i].bar(labels, valores, color=colors, alpha=0.8, edgecolor='black')
    for j, (valor, percent) in enumerate(zip(valores, percents)):
        axes[i].text(j, valor + 2, f'{valor}\n({percent}%)',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[i].set_title(f'Distribuição - {split.capitalize()}\nTotal: {sum(valores)} exemplos',
                     fontsize=12, fontweight='bold', pad=10)
    axes[i].set_ylabel('Número de Exemplos', fontsize=10)
    axes[i].set_ylim(0, max(valores) * 1.15)
    axes[i].grid(axis='y', alpha=0.3, linestyle='--')
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.tight_layout(pad=3.0)
plt.show()


In [ ]:
# observando um registro aleatótio no treino agora com label textual também
import random

# Pegar um índice aleatório no conjunto de treino
random_index = random.randint(0, len(dataset['train']) - 1)

# Buscar o registro aleatório
random_example = dataset['train'][random_index]

print("=== REGISTRO ALEATÓRIO NO TREINO ===")
print(f"Índice: {random_index}")
print(f"Tweet: {random_example['text']}")
print(f"Label: {random_example['label']}")
print(f"Label Textual: {random_example['stance_text']}")

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
import string
from transformers import AutoTokenizer
stop_words = set(stopwords.words('english'))

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

# Ensure necessary NLTK data is downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

def preprocess_tweet(text):
    """
    Função completa de pré-processamento para tweets
    """
    if not isinstance(text, str):
        return ""

    # 1. Converter para minúsculas
    text = text.lower()

    # 2. Remover URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Remover menções de usuário (@username)
    text = re.sub(r'@\w+', '', text)

    # 4. Remover hashtags (mas manter o texto)
    text = re.sub(r'#', '', text)

    # 5. Eliminar algumas palavras especificas que aparecem em muitos tweets porem nao tem significado nenhum (semst,via)
    text = re.sub(r'\b(semst|via)\b', '', text)

    # 6. Remover caracteres especiais e números, mas manter palavras importantes
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 7. Tokenização
    tokens = word_tokenize(text)

    # 8. Juntar tokens de volta em texto
    processed_text = ' '.join(tokens)

    return processed_text

# Aplicar pré-processamento a todo o dataset
def preprocess_batch(examples):
    examples['processed_text'] = [preprocess_tweet(text) for text in examples['text']]
    return examples

print("Aplicando pré-processamento...")
dataset = dataset.map(preprocess_batch, batched=True)

# Mostrar antes e depois do pré-processamento
print("\n=== COMPARAÇÃO ANTES E DEPOIS DO PRÉ-PROCESSAMENTO ===")
for i in range(3):
    original = dataset['train'][i]['text']
    processed = dataset['train'][i]['processed_text']
    print(f"\n--- Exemplo {i+1} ---")
    print(f"ORIGINAL: {original}")
    print(f"PROCESSADO: {processed}")
    print(f"Stance: {dataset['train'][i]['stance_text']}")

In [ ]:
# olhar 3 registros do meu dataset de treino
# Acessando cada campo separadamente
text = dataset['train'][5]['text']
label = dataset['train'][5]['label']
stance_text = dataset['train'][5]['stance_text']
processed_text = dataset['train'][5]['processed_text']

print("Texto original:", text)
print("Label:", label)
print("Stance text:", stance_text)
print("Texto processado:", processed_text)

In [ ]:
# excluir o campo do texto original('text','stance_text')
dataset = dataset.remove_columns(['text','stance_text'])

# MODELAGEM

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# Carregar o tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Função de tokenização corrigida
def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        padding="max_length",
        truncation=True,
        max_length=128,  # importante definir um máximo
        return_attention_mask=True,
        add_special_tokens=True
    )

# Aplicar a tokenização
print("Tokenizando o dataset...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Dataset Tokenizado")

In [ ]:
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]
eval_dataset = tokenized_dataset["validation"]

In [ ]:
print(tokenized_dataset['train'][5])

In [ ]:
# Como abordado no artigo [1] uma abordagem usando uma função de Focal LOSS pode ser melhor que outras abordagens mais tradicionais especialmente em casos de desbalanceamentos bruscos <5%
# abondonado (não deu certo)
# não usar
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=0.5):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
def compute_class_weights(labels):
    class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

# Extract labels from the Hugging Face Dataset
train_labels = train_dataset['label']
class_weights = compute_class_weights(train_labels)
print(f"Distribuição das classes: {np.bincount(train_labels)}")
print(f"Class weights: {class_weights}")

In [ ]:
training_losses = []
validation_losses = []

In [ ]:
# Durante o backpropagation, o gradiente para exemplos da classe rara é multiplicado pelo peso maior, forçando o modelo a prestar mais atenção neles
class CustomTrainer(Trainer):
  def __init__(self, *args, **kwargs):
    super().__init__(*args, **kwargs)
    self.training_losses = []
    self.validation_losses = []
  def log(self, logs, *args, **kwargs):
    super().log(logs, *args, **kwargs)
    epoch = self.state.epoch
    if epoch is not None:
      if 'loss' in logs and 'eval_loss' not in logs:
        self.training_losses.append({
            'epoch': epoch,
            'loss': logs['loss']
            })
      if 'eval_loss' in logs:
        self.validation_losses.append({
            'epoch': epoch,
            'loss': logs['eval_loss']
            })
  def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
      labels = inputs.get("labels")
      outputs = model(**inputs)
      logits = outputs.logits
      # CrossEntropy padrão com class weights
      loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
      loss = loss_fct(logits, labels)
      return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1]
    )

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_none': f1[0],    # índice 0 = none
        'f1_favor': f1[1],   # índice 1 = favor
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./climate_model_stable",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    logging_dir="./logs_stable",
    logging_steps=10,
    warmup_ratio=0.1,
    dataloader_pin_memory=False,
    logging_strategy="steps",
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2  # none, favor
)


In [ ]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
print("Iniciando treinamento...")
trainer.train()

In [ ]:
print(trainer.training_losses[:3])
print(trainer.validation_losses)

In [ ]:
# Função SIMPLES para plotar a curva de aprendizado
def plot_learning_curve(trainer):
    """Plota a curva de aprendizado baseada nos losses capturados"""

    # Extrair dados dos logs
    train_epochs = [log['epoch'] for log in trainer.training_losses]
    train_losses = [log['loss'] for log in trainer.training_losses]

    val_epochs = [log['epoch'] for log in trainer.validation_losses]
    val_losses = [log['loss'] for log in trainer.validation_losses]

    # Criar o gráfico
    plt.figure(figsize=(10, 6))

    # Plot training loss
    if train_epochs:
        plt.plot(train_epochs, train_losses, 'b-', label='Training Loss', linewidth=2, marker='o')

    # Plot validation loss
    if val_epochs:
        plt.plot(val_epochs, val_losses, 'r-', label='Validation Loss', linewidth=2, marker='s')

    plt.title('Curva de Aprendizado')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Só mostrar, não salvar
    plt.show()

# Só isso! Plotar o gráfico
plot_learning_curve(trainer)

In [ ]:
# Avaliar no conjunto de teste
test_results = trainer.evaluate(test_dataset)
print("\n=== RESULTADOS NO TESTE ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")
# Fazer predições no conjunto de teste
predictions = trainer.predict(test_dataset)
print(f"\nPredictions shape: {predictions.predictions.shape}")

In [ ]:
# Avaliação na validação
# Avaliar no conjunto de validacao
test_results = trainer.evaluate(eval_dataset)
print("\n=== RESULTADOS NO TESTE ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")
# Fazer predições no conjunto de teste
predictions = trainer.predict(eval_dataset)
print(f"\nPredictions shape: {predictions.predictions.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Obter predições e labels verdadeiros (validação)
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Matriz de confusão
cm = confusion_matrix(true_labels, preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['none', 'favor'],
            yticklabels=['none', 'favor'])
plt.title('Matriz de Confusão')
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.show()

# Relatório de classificação detalhado
print('\n=== RELATÓRIO DE CLASSIFICAÇÃO ===')
print(classification_report(true_labels, preds,
                            target_names=['none', 'favor']))


In [ ]:
# Mapeamento de labels (binário)
label_mapping = {
    0: "none",   # Nenhum stance
    1: "favor"   # A favor de ações climáticas
}

print('\n=== EXEMPLO DE TRUE STANCE "FAVOR" NO CONJUNTO DE VALIDAÇÃO ===')

for i in range(len(eval_dataset)):
    true_label_id = true_labels[i]
    predicted_label_id = preds[i]

    if true_label_id == 1:  # 1 corresponde a 'favor'
        processed_text = eval_dataset[i]['processed_text']
        true_stance = label_mapping[true_label_id]
        predicted_stance = label_mapping[predicted_label_id]

        print(f'\n--- Exemplo Encontrado (Índice: {i}) ---')
        print(f'Tweet Processado: {processed_text}')
        print(f'Stance Verdadeiro: {true_stance}')
        print(f'Stance Predito: {predicted_stance}')
        if true_label_id != predicted_label_id:
            print('-> **(PREDIÇÃO INCORRETA)**')
        break


In [ ]:
print("\n=== OUTROS EXEMPLOS DE PREDIÇÃO ===")

n_exemplos = 10
for i in range(n_exemplos):
    random_index = np.random.randint(0, len(eval_dataset))
    processed_text = eval_dataset[random_index]['processed_text']
    true_label_id = true_labels[random_index]
    predicted_label_id = preds[random_index]
    true_stance = label_mapping[true_label_id]
    predicted_stance = label_mapping[predicted_label_id]
    print(f"\n--- Exemplo {i+1} ---")
    print(f"Tweet Processado: {processed_text}")
    print(f"Stance Verdadeiro: {true_stance}")
    print(f"Stance Predito: {predicted_stance}")
    if true_label_id != predicted_label_id:
        print("-> **(PREDIÇÃO INCORRETA)**")

# CONSIDERAÇÕES FINAIS DO TRABALHO

*Optamos por remover a classe 'against' do dataset, pois ela representava apenas 3,7% dos exemplos de treino (13 amostras). Essa subrepresentação severa impossibilitava o aprendizado adequado, e o modelo não conseguia classificar nenhum exemplo dessa classe corretamente, nem em validação nem em teste.*

*Com a abordagem binária (none vs favor), o modelo passa a ter duas classes bem distribuídas (~44% none e ~56% favor no treino), permitindo um treinamento mais estável e métricas mais confiáveis. Os f1-scores das classes none e favor foram de 95 e 94 respectivamente, demonstrando excelente desempenho sem overfitting.*


# REFERÊNCIAS

***[1] https://lume.ufrgs.br/handle/10183/259959***